# Task 3 — Define Input Schema and Validate

## Objective

This notebook validates the cleaned Pluralsight dataset against a predefined input schema.

The schema defines:

- Column name
- Expected data type
- Whether the field is nullable
- Allowed values or validation rules

Records that fail validation are written to:

`../data/interim/rejected.csv`

Records that pass validation are written to:

`../data/interim/validated.csv`

## Imports and Data Ingestion

In [1]:
import os
import pandas as pd

# ==============================================================================
# INPUT / OUTPUT DECLARATIONS (Relative path mapping out of notebooks/)
# ==============================================================================
INPUT_CSV_PATH = "../data/interim/cleaned.csv"
VALIDATED_CSV_PATH = "../data/interim/validated.csv"
REJECTED_CSV_PATH = "../data/interim/rejected.csv"

if not os.path.exists(INPUT_CSV_PATH):
    raise FileNotFoundError(f"Missing required interim file source dependency: {INPUT_CSV_PATH}")

df_interim = pd.read_csv(INPUT_CSV_PATH)
print(f"Loaded interim file cleanly. Processing shape container matrix size: {df_interim.shape} items.")

Loaded interim file cleanly. Processing shape container matrix size: (20, 10) items.


## Schema Definition Matrix

| Column Name | Data Type | Nullable | Allowed Values / Constraints |
| :--- | :--- | :--- | :--- |
| **source** | String | No | Must be exactly 'Pluralsight' |
| **category** | String | No | Must be 'AI & Data' or 'Cloud' |
| **title** | String | No | Non-empty character string |
| **author** | String | Yes | Text string tracking writer bylines |
| **publication_date** | String (ISO) | Yes | Must match YYYY-MM-DD standard format |
| **description** | String | Yes | Summary text blocks |
| **tags** | String | Yes | Comma-separated tracking keywords |
| **url** | String | No | Must start with 'https://' |
| **content** | String | No | Full extracted article text content |
| **scraped_at** | String (ISO) | No | Valid ISO execution timestamp |

## Validation Engine

In [3]:
valid_rows = []
rejected_rows = []

# Iterating step-by-step row entries to audit field-level criteria
for idx, row in df_interim.iterrows():
    row_dict = row.to_dict()
    is_valid = True
    failure_reasons = []

    # 1. Validate 'source' (Non-nullable, strict value)
    source_val = str(row_dict.get("source", "")).strip()
    if pd.isna(row_dict.get("source")) or source_val != "Pluralsight":
        is_valid = False
        failure_reasons.append(f"Invalid source property value: '{source_val}'")

    # 2. Validate 'category' (Non-nullable, restricted set values)
    cat_val = str(row_dict.get("category", "")).strip()
    if pd.isna(row_dict.get("category")) or cat_val not in ["AI & Data", "Cloud"]:
        is_valid = False
        failure_reasons.append(f"Category value token breaks allowed bounds: '{cat_val}'")

    # 3. Validate 'title' (Non-nullable text string entity check)
    title_val = str(row_dict.get("title", "")).strip()
    if pd.isna(row_dict.get("title")) or title_val == "" or title_val == "nan":
        is_valid = False
        failure_reasons.append("Title attribute constraint broken: value is blank or missing")

    # 4. Validate 'url' (Non-nullable network scheme filter)
    url_val = str(row_dict.get("url", "")).strip()
    if pd.isna(row_dict.get("url")) or not url_val.startswith("https://"):
        is_valid = False
        failure_reasons.append("URL validation structure dropped out from network base domain scheme filters")

    # 5. Validate 'content' (Non-nullable string requirement tracking)
    content_val = str(row_dict.get("content", "")).strip()
    if pd.isna(row_dict.get("content")) or content_val == "" or content_val == "nan":
        is_valid = False
        failure_reasons.append("Article body tracking content returned string length empty exception flags")

    # Routing processing flows safely based on boolean evaluations
    if is_valid:
        valid_rows.append(row_dict)
    else:
        row_dict["rejection_reason"] = "; ".join(failure_reasons)
        rejected_rows.append(row_dict)

# Reassemble tracking containers
df_validated = pd.DataFrame(valid_rows)
df_rejected = pd.DataFrame(rejected_rows)

print("=========================================================================")
print("SCHEMA ROUTING EVALUATION COMPLETE SUMMARY")
print("=========================================================================")
print(f"Total Rows Passing Schema Validation:  {len(df_validated)}")
print(f"Total Rows Routed to Rejection Bucket: {len(df_rejected)}")

SCHEMA ROUTING EVALUATION COMPLETE SUMMARY
Total Rows Passing Schema Validation:  20
Total Rows Routed to Rejection Bucket: 0


## Checkpoint Export Files Flush

In [4]:
# Save clean passing collection streams
if not df_validated.empty:
    df_validated.to_csv(VALIDATED_CSV_PATH, index=False, encoding="utf-8")
else:
    # Build empty DataFrame structure placeholders matching expected layouts to prevent file dependencies dropping completely downstream
    pd.DataFrame(columns=df_interim.columns).to_csv(VALIDATED_CSV_PATH, index=False)

# Save rejected matrix outputs containing custom validation logic logging data fields
if not df_rejected.empty:
    df_rejected.to_csv(REJECTED_CSV_PATH, index=False, encoding="utf-8")
else:
    # Build clean structure frameworks to guarantee safe reading behaviors down the pipe
    pd.DataFrame(columns=list(df_interim.columns) + ["rejection_reason"]).to_csv(REJECTED_CSV_PATH, index=False)

print("All validation split states written accurately to target local repositories:")
print(f" - Valid Rows Target:   {VALIDATED_CSV_PATH}")
print(f" - Rejected Rows Target: {REJECTED_CSV_PATH}")


All validation split states written accurately to target local repositories:
 - Valid Rows Target:   ../data/interim/validated.csv
 - Rejected Rows Target: ../data/interim/rejected.csv
